# LoRA From Scratch

LoRA, or Low-Rank Adaptation, is one of the most important parameter-efficient fine-tuning methods. Instead of updating the full weight matrix, LoRA freezes the original weights and learns a small low-rank update.

This notebook builds LoRA directly in PyTorch so the PEFT and QLoRA notebooks feel transparent rather than magical.

## What You Will Learn

By the end, you should understand:

- why full fine-tuning is expensive,
- how freezing changes trainable parameter count,
- the LoRA equation,
- rank, alpha, and scaling,
- how to wrap a linear layer with LoRA,
- how adapter merging works,
- and why LoRA is useful for large LLMs.

## 1. Full Fine-Tuning vs LoRA

A transformer contains many large linear layers. Full fine-tuning updates all of their weights. LoRA freezes the base weights and trains a smaller update:

```text
y = xW^T + scale * x(BA)^T
```

Where:

- `W` is the frozen base weight,
- `A` projects from input dimension to low rank `r`,
- `B` projects from low rank `r` back to output dimension,
- `scale = alpha / r`.

If `r` is small, the trainable update is much cheaper than the original matrix.

In [ ]:
import math

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## 2. Parameter Count Intuition

For a linear layer with shape `out_features x in_features`, full fine-tuning trains `out_features * in_features` parameters. LoRA trains `r * in_features + out_features * r` parameters.

In [ ]:
def lora_parameter_count(in_features, out_features, rank):
    full = in_features * out_features
    lora = rank * in_features + out_features * rank
    return {
        "in_features": in_features,
        "out_features": out_features,
        "rank": rank,
        "full_params": full,
        "lora_params": lora,
        "trainable_percent": 100 * lora / full,
    }

rows = [lora_parameter_count(4096, 4096, rank) for rank in [4, 8, 16, 32, 64]]
pd.DataFrame(rows)

## 3. A LoRA Linear Layer

This implementation keeps a frozen base linear layer and adds two trainable matrices. We initialize `lora_b` to zeros so the LoRA path starts as no-op. That means the wrapped layer initially behaves exactly like the base layer.

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base_layer, rank=8, alpha=16, dropout=0.0):
        super().__init__()
        if not isinstance(base_layer, nn.Linear):
            raise TypeError("base_layer must be nn.Linear")
        self.base_layer = base_layer
        self.rank = rank
        self.alpha = alpha
        self.scale = alpha / rank
        self.dropout = nn.Dropout(dropout)

        in_features = base_layer.in_features
        out_features = base_layer.out_features
        self.lora_a = nn.Parameter(torch.empty(rank, in_features))
        self.lora_b = nn.Parameter(torch.zeros(out_features, rank))
        nn.init.kaiming_uniform_(self.lora_a, a=math.sqrt(5))

        for parameter in self.base_layer.parameters():
            parameter.requires_grad = False

    def forward(self, x):
        base_output = self.base_layer(x)
        lora_hidden = F.linear(self.dropout(x), self.lora_a)
        lora_output = F.linear(lora_hidden, self.lora_b)
        return base_output + self.scale * lora_output

    def merged_weight(self):
        delta = self.scale * (self.lora_b @ self.lora_a)
        return self.base_layer.weight + delta

base = nn.Linear(16, 8)
lora = LoRALinear(base, rank=4, alpha=8)
print(lora)

## 4. Check Initial No-Op Behavior

Because `lora_b` starts at zero, the adapter update is zero at initialization. This is important because adding LoRA should not randomly destroy the base model's behavior at step zero.

In [ ]:
x = torch.randn(3, 16)
with torch.inference_mode():
    base_out = base(x)
    lora_out = lora(x)
print(torch.max(torch.abs(base_out - lora_out)).item())

## 5. Train Only LoRA Parameters

We will create a toy regression problem. The base layer is frozen. The only trainable parameters are `lora_a` and `lora_b`. This mirrors how LoRA adapts large transformer layers without updating the whole model.

In [ ]:
def count_parameters(module):
    total = sum(parameter.numel() for parameter in module.parameters())
    trainable = sum(parameter.numel() for parameter in module.parameters() if parameter.requires_grad)
    return total, trainable

total, trainable = count_parameters(lora)
print(f"total parameters: {total}")
print(f"trainable parameters: {trainable}")
for name, parameter in lora.named_parameters():
    print(f"{name:24s} trainable={parameter.requires_grad} shape={tuple(parameter.shape)}")

In [ ]:
in_features = 16
out_features = 8
num_samples = 512

true_weight = torch.randn(out_features, in_features) * 0.5
true_bias = torch.randn(out_features) * 0.1
features = torch.randn(num_samples, in_features)
targets = F.linear(features, true_weight, true_bias)

base_layer = nn.Linear(in_features, out_features)
lora_model = LoRALinear(base_layer, rank=4, alpha=8, dropout=0.0)
optimizer = torch.optim.AdamW([p for p in lora_model.parameters() if p.requires_grad], lr=0.05)

losses = []
for step in range(250):
    predictions = lora_model(features)
    loss = F.mse_loss(predictions, targets)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

print(f"initial_loss: {losses[0]:.4f}")
print(f"final_loss: {losses[-1]:.4f}")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(losses)
plt.xlabel("step")
plt.ylabel("MSE loss")
plt.title("LoRA-only training on a toy regression problem")
plt.grid(alpha=0.3)
plt.tight_layout()

## 6. Merging LoRA Weights

For inference, LoRA can be kept as a separate adapter or merged into the base weights. Merging computes:

```text
W_merged = W_base + scale * B @ A
```

After merging, inference can use a normal linear layer. This is convenient for deployment, but it removes the easy ability to swap adapters.

In [ ]:
def merge_lora_linear(lora_layer):
    merged = nn.Linear(
        lora_layer.base_layer.in_features,
        lora_layer.base_layer.out_features,
        bias=lora_layer.base_layer.bias is not None,
    )
    with torch.no_grad():
        merged.weight.copy_(lora_layer.merged_weight())
        if lora_layer.base_layer.bias is not None:
            merged.bias.copy_(lora_layer.base_layer.bias)
    return merged

merged_model = merge_lora_linear(lora_model)
with torch.inference_mode():
    adapter_output = lora_model(features[:16])
    merged_output = merged_model(features[:16])
print(torch.max(torch.abs(adapter_output - merged_output)).item())

## 7. Which Transformer Layers Get LoRA?

In LLM fine-tuning, LoRA is commonly applied to attention and MLP projection layers. Names vary by model family, but common target modules include:

| Target | Role |
|---|---|
| `q_proj` | Query projection in attention |
| `k_proj` | Key projection in attention |
| `v_proj` | Value projection in attention |
| `o_proj` | Attention output projection |
| `gate_proj` | MLP gate projection in SwiGLU-style blocks |
| `up_proj` | MLP expansion projection |
| `down_proj` | MLP contraction projection |

A common first setting is target attention projections, then expand to MLP projections if quality needs improve and memory allows.

## 8. Rank and Alpha Intuition

| Setting | Effect | Trade-off |
|---|---|---|
| Higher rank `r` | More expressive adapter | More memory and compute |
| Lower rank `r` | Cheaper adapter | May underfit complex changes |
| Higher alpha | Larger adapter update scale | Can destabilize if too high |
| Dropout | Regularizes adapter path | Can slow fitting on small data |

Typical starting points: `r=8` or `r=16`, `alpha=16` or `alpha=32`, dropout around `0.05` for instruction data.

## Summary

LoRA freezes the base model and trains a low-rank update. This gives most of the practical benefit of fine-tuning while training far fewer parameters. You now saw the actual math, parameter savings, no-op initialization, adapter-only training, and merging.

Next, we will use Hugging Face PEFT to apply LoRA to a real language model fine-tuning workflow.